In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
from plot_utils import topk_barplot, cumsum_plot

In [ ]:
sample_size = 25

In [ ]:
current_dir = os.getcwd()

data_dir = "tech_challenge-data-us_flights_ml"

In [ ]:
flights_csv = "flights.csv"
df_flights = pd.read_csv(os.path.join(current_dir, data_dir, flights_csv))

In [ ]:
df_flights.info()

In [ ]:
df_flights_considered = df_flights.loc[
    (df_flights["CANCELLED"] == 0) & (df_flights["DIVERTED"] == 0) & (df_flights["MONTH"] != 10),
    [
        "YEAR", "MONTH", "DAY", "DAY_OF_WEEK",
        "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER",
        "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DISTANCE",
        "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "SCHEDULED_ARRIVAL",
        "ARRIVAL_DELAY"
    ]
]

In [ ]:
del df_flights
gc.collect()

In [ ]:
df_flights_considered["HOUR"] = df_flights_considered["SCHEDULED_DEPARTURE"] // 100

wd_mapping = {
    1: "MON",
    2: "TUE",
    3: "WED",
    4: "THU",
    5: "FRI",
    6: "SAT",
    7: "SUN",
}
df_flights_considered["DAY_OF_WEEK_ABBR"] = pd.Categorical(
    df_flights_considered["DAY_OF_WEEK"],
    categories=wd_mapping.keys(),
    ordered=True
).rename_categories(wd_mapping)

df_flights_considered["ROUTE"] = df_flights_considered["ORIGIN_AIRPORT"].astype(str) + "-" + df_flights_considered["DESTINATION_AIRPORT"].astype(str)

tol_min = 15
df_flights_considered["DELAYED"] = df_flights_considered["ARRIVAL_DELAY"] >= tol_min

In [ ]:
df_flights_considered.info()

In [ ]:
df_flights_considered.sample(n=sample_size)

## Sampled dataset

In [ ]:
rng = np.random.default_rng(seed=42)

n_rows, _ = df_flights_considered.shape
sampled_dataset_mask = np.zeros(n_rows, dtype=bool)

sampled_dataset_size = 100_000
sampled_dataset_indices = rng.choice(n_rows, size=sampled_dataset_size, replace=False)

sampled_dataset_mask[sampled_dataset_indices] = True

In [ ]:
df_flights_sampled = df_flights_considered[sampled_dataset_mask].copy(deep=True)

In [ ]:
df_flights_sampled.info()

In [ ]:
df_flights_sampled.sample(n=sample_size)

In [ ]:
df_flights_sampled["ORIGIN_AIRPORT"] = df_flights_sampled["ORIGIN_AIRPORT"].astype(str)
df_flights_sampled["DESTINATION_AIRPORT"] = df_flights_sampled["DESTINATION_AIRPORT"].astype(str)
df_flights_sampled["SCHEDULED_TIME"] = df_flights_sampled["SCHEDULED_TIME"].astype(int)

In [ ]:
plot_opts = dict(
    kind="bar",
    stacked=True,
    color=["deepskyblue", "gainsboro"]
)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["MONTH", "DELAYED"]

df_flights_considered[cols].value_counts().unstack().reindex(range(1, 13), fill_value=0).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack().reindex(range(1, 13), fill_value=0).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["DAY", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["DAY_OF_WEEK_ABBR", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["HOUR", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["ORIGIN_AIRPORT", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["DESTINATION_AIRPORT", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["ROUTE", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["AIRLINE", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["FLIGHT_NUMBER", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["TAIL_NUMBER", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
del df_flights_considered
gc.collect()